# 01_preparacao_demografia

Preparacao da base de demografia do CAF para modelagem de ML (PU Learning).


## 1) Introdução

Este notebook lê a base demográfica bruta, aplica limpeza mínima, cria variáveis administrativas úteis para modelagem, calcula idade em 2025 e salva a base final em parquet.

Resultado esperado: dataframe final por `cpf_agricultor`, com dtypes otimizados e pronto para etapas de modelagem.

## 2) Imports e configurações

Nesta etapa definimos caminhos, importamos metadados e ajustamos parâmetros de execução.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Raiz do projeto (funciona executando a partir da raiz ou da pasta notebooks)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Permite import de módulos locais (ex.: src/metadata_caf.py)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.metadata_caf import METADATA_CAF

DEMOGRAFIA_PATH = ROOT / "dados" / "dados_brutos" / "a_tcc_v2_pessoa_demografia_compilado.parquet"
OUTPUT_PATH = ROOT / "dados" / "dados_processados" / "df_caf_demografia.parquet"

# Use fração apenas para desenvolvimento rápido. Em produção, deixe None.
SAMPLE_FRAC = None  # Ex.: 0.1

if not DEMOGRAFIA_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DEMOGRAFIA_PATH}")

print(f"ROOT: {ROOT}")
print(f"DEMOGRAFIA_PATH: {DEMOGRAFIA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

## 3) Carregamento dos dados

Lemos apenas as colunas necessárias para reduzir uso de memória.

In [ ]:
COLS_INTERESSE_CAF_DEMOGRAFIA = [
    "ibge",
    "cpf_agricultor",
    "id_familia",
    "id_tipo_membro_familiar",
    "dt_nascimento",
    "id_tipo_estado_civil",
    "id_sexo",
    "id_tipo_etnia",
    "st_trabalha_ufpr",
    "cd_sipra",
    "enquadramento_renda",
]

df_caf_demografia = pd.read_parquet(
    DEMOGRAFIA_PATH,
    columns=COLS_INTERESSE_CAF_DEMOGRAFIA,
    engine="pyarrow",
)

if SAMPLE_FRAC is not None:
    df_caf_demografia = df_caf_demografia.sample(frac=SAMPLE_FRAC, random_state=42)

print(f"Shape inicial: {df_caf_demografia.shape}")
display(df_caf_demografia.head(3))

## 4) Limpeza mínima e padronização inicial

Padronizamos tipos de identificadores e códigos, tratamos categorias "não informado" e criamos a flag de assentamento a partir de `cd_sipra`.

In [ ]:
# Identificadores como string
for col in ["ibge", "cpf_agricultor", "id_familia"]:
    df_caf_demografia[col] = df_caf_demografia[col].astype("string")

# Códigos categóricos compactos
df_caf_demografia["id_tipo_membro_familiar"] = df_caf_demografia["id_tipo_membro_familiar"].astype("int8")
df_caf_demografia["id_sexo"] = df_caf_demografia["id_sexo"].astype("int8")

# Tratamento de "não informado" antes do cast
df_caf_demografia["id_tipo_estado_civil"] = (
    df_caf_demografia["id_tipo_estado_civil"].fillna(7).astype("int8")
)
df_caf_demografia["id_tipo_etnia"] = (
    df_caf_demografia["id_tipo_etnia"].fillna(6).astype("int8")
)

# Flag de trabalho na UFPRAF como 0/1
df_caf_demografia["st_trabalha_ufpr"] = (
    df_caf_demografia["st_trabalha_ufpr"].fillna(False).astype("int8")
)

# cd_sipra é código (string). Criamos flag binária de assentado.
df_caf_demografia["cd_sipra"] = df_caf_demografia["cd_sipra"].astype("string")
df_caf_demografia["st_assentado_sipra"] = (
    df_caf_demografia["cd_sipra"].notna()
    & (df_caf_demografia["cd_sipra"].str.strip() != "")
).astype("int8")

df_caf_demografia = df_caf_demografia.drop(columns=["cd_sipra"])

# Checagens leves
display(df_caf_demografia[["id_tipo_estado_civil", "id_tipo_etnia", "st_assentado_sipra"]].isna().sum())
display(df_caf_demografia["st_assentado_sipra"].value_counts(dropna=False))

## 5) Features administrativas: enquadramento de renda

Tratamos nulos, geramos dummies e consolidamos uma linha por `cpf_agricultor`.

In [ ]:
# Normaliza a coluna e trata ausências
df_caf_demografia["enquadramento_renda"] = (
    df_caf_demografia["enquadramento_renda"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "A/C": "AC"})
    .fillna("Desconhecido")
)

print("Distribuição de enquadramento_renda:")
display(df_caf_demografia["enquadramento_renda"].value_counts(dropna=False).sort_index())

# Dummies com colunas esperadas para estabilidade do pipeline
enq_dummies = pd.get_dummies(df_caf_demografia["enquadramento_renda"], dtype="int8")
for col in ["A", "AC", "B", "Desconhecido", "V"]:
    if col not in enq_dummies.columns:
        enq_dummies[col] = 0

enq_dummies = enq_dummies[["A", "AC", "B", "Desconhecido", "V"]]

linhas_antes_dedup = len(df_caf_demografia)

df_caf_por_cpf = pd.concat([df_caf_demografia, enq_dummies], axis=1)
df_caf_por_cpf = (
    df_caf_por_cpf
    .drop_duplicates(subset=["cpf_agricultor"], keep="first")
    .drop(columns=["enquadramento_renda"])
)

linhas_depois_dedup = len(df_caf_por_cpf)

print(f"Linhas antes do dedup por CPF: {linhas_antes_dedup:,}")
print(f"Linhas após o dedup por CPF: {linhas_depois_dedup:,}")
print(f"CPFs distintos: {df_caf_por_cpf['cpf_agricultor'].nunique():,}")

## 6) Datas e idade

Convertendo `dt_nascimento`, calculamos `idade_2025`, filtramos para faixa etária de adultos (18 a 100 anos) e criamos a flag `jovem` (18 a 29).

In [ ]:
linhas_antes_data = len(df_caf_por_cpf)

# Remove vazios de data e converte para datetime
df_caf_por_cpf = df_caf_por_cpf.dropna(subset=["dt_nascimento"]).copy()
df_caf_por_cpf["dt_nascimento"] = pd.to_datetime(
    df_caf_por_cpf["dt_nascimento"],
    errors="coerce",
    dayfirst=True,
)

# Remove datas inválidas (NaT)
df_caf_por_cpf = df_caf_por_cpf.dropna(subset=["dt_nascimento"]).copy()

data_ref = pd.Timestamp("2025-12-31")

# Idade completa em anos na data de referência
df_caf_por_cpf["idade_2025"] = (
    data_ref.year - df_caf_por_cpf["dt_nascimento"].dt.year
    - (
        (data_ref.month < df_caf_por_cpf["dt_nascimento"].dt.month)
        | (
            (data_ref.month == df_caf_por_cpf["dt_nascimento"].dt.month)
            & (data_ref.day < df_caf_por_cpf["dt_nascimento"].dt.day)
        )
    )
)

linhas_antes_filtro_idade = len(df_caf_por_cpf)
df_caf_por_cpf = df_caf_por_cpf[df_caf_por_cpf["idade_2025"].between(0, 120)].copy()
linhas_depois_filtro_idade = len(df_caf_por_cpf)

print(f"Linhas antes do tratamento de data: {linhas_antes_data:,}")
print(f"Linhas antes do filtro de idade (0-120): {linhas_antes_filtro_idade:,}")
print(f"Linhas após filtro de idade (0-120): {linhas_depois_filtro_idade:,}")

# Cast final de idade + flag jovem
df_caf_por_cpf["idade_2025"] = df_caf_por_cpf["idade_2025"].astype("int8")
df_caf_por_cpf["jovem"] = (
    df_caf_por_cpf["idade_2025"].between(18, 29, inclusive="both").astype("int8")
)

# dt_nascimento não é mais necessária para modelagem
df_caf_por_cpf = df_caf_por_cpf.drop(columns=["dt_nascimento"]).reset_index(drop=True)

display(df_caf_por_cpf[["idade_2025", "jovem"]].head())

## 7) Padronização final de dtypes

Garantimos consistência dos tipos para economizar memória e evitar problemas em etapas posteriores do pipeline.

In [ ]:
# Strings para chaves
for col in ["ibge", "cpf_agricultor", "id_familia"]:
    df_caf_por_cpf[col] = df_caf_por_cpf[col].astype("string")

# int8 para códigos, flags e dummies
int8_cols = [
    "id_tipo_membro_familiar",
    "id_tipo_estado_civil",
    "id_sexo",
    "id_tipo_etnia",
    "st_trabalha_ufpr",
    "st_assentado_sipra",
    "A",
    "AC",
    "B",
    "Desconhecido",
    "V",
    "idade_2025",
    "jovem",
]

for col in int8_cols:
    if col in df_caf_por_cpf.columns:
        df_caf_por_cpf[col] = df_caf_por_cpf[col].astype("int8")

display(df_caf_por_cpf.dtypes.sort_index())

## 8) Validação final

Checagens rápidas de estrutura, nulos e distribuição em variáveis chave.

In [ ]:
print(f"Shape final: {df_caf_por_cpf.shape}")

# Diagnóstico de nulos em campos críticos
display(df_caf_por_cpf[["id_tipo_etnia", "id_tipo_estado_civil", "st_assentado_sipra"]].isna().sum())

# Distribuições rápidas
display(df_caf_por_cpf["st_assentado_sipra"].value_counts(dropna=False).sort_index())
display(df_caf_por_cpf["id_tipo_etnia"].value_counts(dropna=False).sort_index())
display(df_caf_por_cpf[["A", "AC", "B", "Desconhecido", "V"]].sum().to_frame("contagem"))

# Exemplo de leitura com metadados de etnia
dist_etnia = (
    df_caf_por_cpf["id_tipo_etnia"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("quantidade")
    .to_frame()
)
dist_etnia["etnia"] = dist_etnia.index.map(METADATA_CAF["id_tipo_etnia"])
dist_etnia["percentual"] = (dist_etnia["quantidade"] / dist_etnia["quantidade"].sum() * 100).round(2)
display(dist_etnia[["etnia", "quantidade", "percentual"]])

# Estrutura final
df_caf_por_cpf.info()
display(df_caf_por_cpf.head(5))

## 9) Persistência

Salvamos o dataframe final em parquet para uso nas próximas etapas do pipeline.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_caf_por_cpf.to_parquet(OUTPUT_PATH, index=False)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Shape salvo: {df_caf_por_cpf.shape}")
print(f"Total de colunas: {len(df_caf_por_cpf.columns)}")
display(df_caf_por_cpf.columns.to_list())